# LLM-as-a-Judge v2: Contextual Attachment Scoring

This notebook tests the **improved scoring approach** that addresses the sparse high-attachment distribution (v1 had only 1.3% with scores ≥5).

## Key Improvements in v2

1. **Contextual Attachment Scoring**: Judge sees BOTH assistant response AND user reply (not just user reply in isolation)
2. **Single-Pass Strict Scoring**: Uses the base contextual rubric only (no lenient aggregation)
3. **Cleaner Implementation**: Simplified from dual-pass to single contextual evaluation

## Scoring Scheme

- **Empathy Score (T):** 1-7, applied to `llm_response`
- **Attachment Score (Y):** 1-7, applied to `user_reply` WITH `llm_response` context

## Judge Model

Using **Llama 3.1 8B Instant** via Groq API

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
import time
from groq import Groq

# Add scripts directory to path
sys.path.append('../scripts')

# Load environment variables
load_dotenv()

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 200)

# Set style for plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries imported successfully")

In [ ]:
# Test API connection
try:
    groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))
    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": "Hello, respond with just 'OK'"}],
        max_tokens=5
    )
    print("✅ Groq/Llama API Success!")
    print(f"Response: {response.choices[0].message.content}")
except Exception as e:
    print(f"❌ Groq/Llama API Error: {e}")
    print("\nMake sure you have set GROQ_API_KEY in your .env file")

## 1. Load Sample Dataset

In [ ]:
# Load the sample preprocessed dataset
sample_path = '../data/filtered/wildchat_sample_preprocessed_v2.csv'

print(f"Loading sample dataset from: {sample_path}")
df_sample = pd.read_csv(sample_path)

print(f"\nDataset shape: {df_sample.shape}")
print(f"\nColumns: {list(df_sample.columns)}")
print(f"\nFirst few rows:")
df_sample.head()

## 2. Import Scoring Function (v2)

This uses `score_conversations2.py` which implements:
- Contextual attachment scoring (assistant response + user reply)
- Strict single-pass evaluation
- Direct assignment (no aggregation)

In [ ]:
# Import the contextual scoring function
from score_conversations2 import score_conversations

print("✓ Imported scoring function from score_conversations2.py")
print("\nThis version:")
print("  - Uses BOTH assistant response + user reply for attachment scoring (contextual)")
print("  - Single-pass strict evaluation (no lenient aggregation)")
print("  - Attachment score = strict contextual score (1-7)")

## 3. Score a Small Subset (5 pairs)

Test the new approach on 5 turn pairs first

In [ ]:
# Score a small subset first (5 turn pairs) to test
print("Scoring a small subset (5 turn pairs) with v2 approach...")
print("="*70)

df_subset = df_sample.head(5).copy()
df_subset_scored = score_conversations(df_subset, verbose=True)

In [ ]:
# Examine the scored results
print("Scored Subset Results (v2 - Contextual):")
print("="*70)
print(df_subset_scored[['turn_pair_id', 'model', 'empathy_score', 'attachment_score']])

print("\n" + "="*70)
print("Score Distribution:")
print(f"\nEmpathy Scores:")
print(df_subset_scored['empathy_score'].value_counts().sort_index())
print(f"\nAttachment Scores (Contextual):")
print(df_subset_scored['attachment_score'].value_counts().sort_index())

## 4. Score Subset of 60 Conversations

Testing the contextual scoring on 60 turn pairs (instead of all 223).

**Note**: This makes ~120 API calls (60 × 2: empathy + attachment) and should take ~3-5 minutes.

In [ ]:
# Score a SUBSET of 60 turn pairs with contextual approach
TEST_SIZE = 60

print(f"Testing contextual scoring on {TEST_SIZE} turn pairs...")
print("This should take ~3-5 minutes (2 API calls per pair).")
print("="*70)

# Take first 60 conversations
df_test = df_sample.head(TEST_SIZE).copy()

print(f"\nScoring {len(df_test)} turn pairs...")
df_sample_scored_v2 = score_conversations(df_test, verbose=True)

In [ ]:
# Save the scored subset (v2 with contextual scoring)
output_path = '../data/scores/wildchat_60sample_scored_v2_contextual.csv'

# Create directory if it doesn't exist
os.makedirs('../data/scores', exist_ok=True)

print(f"Saving scored subset to: {output_path}")
df_sample_scored_v2.to_csv(output_path, index=False)

print(f"✓ Saved successfully!")
print(f"  Rows: {len(df_sample_scored_v2)}")
print(f"  Columns: {len(df_sample_scored_v2.columns)}")
print(f"  File size: {os.path.getsize(output_path) / (1024):.2f} KB")

## 5. Analyze v2 Score Distributions

In [ ]:
# Check for missing scores
print("Data Quality Check (v2):")
print("="*70)
print(f"Total turn pairs: {len(df_sample_scored_v2)}")
print(f"\nMissing empathy scores: {df_sample_scored_v2['empathy_score'].isna().sum()}")
print(f"Missing attachment (strict): {df_sample_scored_v2['attachment_score_strict'].isna().sum()}")
print(f"Missing attachment (lenient): {df_sample_scored_v2['attachment_score_lenient'].isna().sum()}")
print(f"Missing attachment (final): {df_sample_scored_v2['attachment_score'].isna().sum()}")

In [ ]:
# Score distributions (v2)
print("\nEmpathy Score Distribution (v2):")
print("="*70)
empathy_counts = df_sample_scored_v2['empathy_score'].value_counts().sort_index()
empathy_pct = (empathy_counts / len(df_sample_scored_v2) * 100).round(2)

print(f"{'Score':>6} {'Count':>10} {'Percentage':>12}")
print("-"*30)
for score in range(1, 8):
    count = empathy_counts.get(score, 0)
    pct = empathy_pct.get(score, 0.0)
    print(f"{score:>6} {count:>10} {pct:>11.2f}%")
print("-"*30)
print(f"{'Total':>6} {len(df_sample_scored_v2):>10} {'100.00%':>12}")

print(f"\nSummary Statistics:")
print(df_sample_scored_v2['empathy_score'].describe())

In [ ]:
# Attachment distribution - contextual scoring
print("\nAttachment Score Distribution (v2 - Contextual):")
print("="*70)

counts = df_sample_scored_v2['attachment_score'].value_counts().sort_index()
pct = (counts / len(df_sample_scored_v2) * 100).round(2)

print(f"{'Score':>6} {'Count':>10} {'Percentage':>12}")
print("-"*30)
for score in range(1, 8):
    count = counts.get(score, 0)
    p = pct.get(score, 0.0)
    print(f"{score:>6} {count:>10} {p:>11.2f}%")
print("-"*30)
print(f"{'Total':>6} {len(df_sample_scored_v2):>10} {'100.00%':>12}")

# High attachment (>=5)
high_att = df_sample_scored_v2['attachment_score'].ge(5).sum()
high_att_pct = (high_att / len(df_sample_scored_v2) * 100).round(2)
print(f"\nHigh Attachment (≥5): {high_att} ({high_att_pct}%)")

print(f"\nSummary Stats:")
print(f"  Mean: {df_sample_scored_v2['attachment_score'].mean():.2f}")
print(f"  Median: {df_sample_scored_v2['attachment_score'].median():.1f}")
print(f"  Std: {df_sample_scored_v2['attachment_score'].std():.2f}")

In [ ]:
# Visualize attachment distribution
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

ax.hist(df_sample_scored_v2['attachment_score'].dropna(), bins=np.arange(0.5, 8.5, 1),
        color='steelblue', edgecolor='black', alpha=0.7)
ax.set_title('Attachment Score Distribution (v2 - Contextual)', fontsize=14, fontweight='bold')
ax.set_xlabel('Score (1-7)', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_xticks(range(1, 8))

mean_val = df_sample_scored_v2['attachment_score'].mean()
ax.axvline(mean_val, color='red', linestyle='--',
           label=f'Mean: {mean_val:.2f}')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Summary

### Contextual Scoring Approach (v2)

**Main Goal**: Improve attachment score distribution by providing context to the judge

**Key Changes from v1**:
1. **Contextual evaluation**: Judge sees BOTH assistant response + user reply (not just user reply)
2. **Single-pass scoring**: Simplified from dual-pass aggregation to direct contextual evaluation
3. **Better signal detection**: Context helps identify attachment indicators more accurately

### Test Results (60 conversations)

Run the analysis cells above to see:
- Attachment score distribution
- High attachment (≥5) frequency
- Comparison to v1 (user reply only)

### Files Generated

- `data/scores/wildchat_60sample_scored_v2_contextual.csv`
  - Test of contextual scoring approach
  - Columns: `empathy_score`, `attachment_score`

### Next Steps

1. ✅ Test contextual scoring on 60 pairs
2. Evaluate distribution improvements vs v1
3. If successful, scale to full dataset
4. Generate embeddings for propensity score matching
5. Perform causal analysis

## 6.5 Distribution Analysis

Analyzing the score distribution from contextual scoring

In [ ]:
# DISTRIBUTION ANALYSIS: Contextual scoring results
print("="*70)
print("ATTACHMENT SCORE DISTRIBUTION ANALYSIS (V2 - Contextual)")
print("="*70)

# Calculate distribution metrics
score_counts = df_sample_scored_v2['attachment_score'].value_counts().sort_index()
total_valid = df_sample_scored_v2['attachment_score'].notna().sum()

print(f"\nScore Distribution:")
print(f"{'Score':>6} {'Count':>10} {'Percentage':>12}")
print("-"*30)
for score in range(1, 8):
    count = score_counts.get(score, 0)
    pct = (count / total_valid * 100) if total_valid > 0 else 0
    print(f"{score:>6} {count:>10} {pct:>11.2f}%")

# Calculate score=4 frequency
score_4_count = score_counts.get(4, 0)
score_4_pct = (score_4_count / total_valid * 100) if total_valid > 0 else 0

print(f"\n" + "="*70)
print("SCORE=4 FREQUENCY:")
print("="*70)
print(f"Score=4: {score_4_count} / {total_valid} ({score_4_pct:.1f}%)")

# Data split for causal analysis
control = (df_sample_scored_v2['attachment_score'] <= 3).sum()
excluded = score_4_count
treatment = (df_sample_scored_v2['attachment_score'] >= 5).sum()

print(f"\n" + "="*70)
print("DATA SPLIT FOR CAUSAL ANALYSIS:")
print("="*70)
print(f"Control (≤3):     {control:3d} ({control/total_valid*100:5.1f}%)")
print(f"EXCLUDED (=4):    {excluded:3d} ({excluded/total_valid*100:5.1f}%)")
print(f"Treatment (≥5):   {treatment:3d} ({treatment/total_valid*100:5.1f}%)")
print(f"Usable data:      {control + treatment:3d} ({(control + treatment)/total_valid*100:5.1f}%)")

# Summary statistics
print(f"\n" + "="*70)
print("SUMMARY STATISTICS:")
print("="*70)
print(f"Mean:   {df_sample_scored_v2['attachment_score'].mean():.2f}")
print(f"Median: {df_sample_scored_v2['attachment_score'].median():.1f}")
print(f"Std:    {df_sample_scored_v2['attachment_score'].std():.2f}")
print(f"Min:    {df_sample_scored_v2['attachment_score'].min():.0f}")
print(f"Max:    {df_sample_scored_v2['attachment_score'].max():.0f}")
print("="*70)